In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path

# Expected dataset path
zip_path = Path("/content/drive/MyDrive/wind_dataset.zip")

print(f"Exists        : {zip_path.exists()}")
print(f"Absolute Path : {zip_path.resolve()}")

if zip_path.exists():
    size_gb = zip_path.stat().st_size / (1024**3)
    print(f"File Size (GB): {size_gb:.4f}")
else:
    print("ERROR: ZIP file not found.")

Exists        : True
Absolute Path : /content/drive/MyDrive/wind_dataset.zip
File Size (GB): 0.0419


In [ ]:
import zipfile
from pathlib import Path

zip_path = Path("/content/drive/MyDrive/wind_dataset.zip")

with zipfile.ZipFile(zip_path, 'r') as z:
    file_list = z.infolist()

    print(f"Total Files Inside ZIP: {len(file_list)}")
    print("-" * 60)

    total_uncompressed = 0

    for i, file in enumerate(file_list):
        size_mb = file.file_size / (1024**2)
        total_uncompressed += file.file_size

        print(f"[{i}] {file.filename}")
        print(f"     Uncompressed Size: {size_mb:.2f} MB")
        print(f"     Compressed Size  : {file.compress_size / (1024**2):.2f} MB")
        print("-" * 60)

    print(f"Total Uncompressed Size: {total_uncompressed / (1024**3):.4f} GB")

Total Files Inside ZIP: 1
------------------------------------------------------------
[0] data (1).csv
     Uncompressed Size: 196.68 MB
     Compressed Size  : 42.95 MB
------------------------------------------------------------
Total Uncompressed Size: 0.1921 GB


In [ ]:
import zipfile
from pathlib import Path

zip_path = Path("/content/drive/MyDrive/wind_dataset.zip")

extract_dir = Path("/content/wind_dataset")

extract_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_dir)

print("Extraction Complete.")
print(f"Extracted To: {extract_dir}")

print("\nFiles:")
for p in extract_dir.iterdir():
    print("-", p.name)

Extraction Complete.
Extracted To: /content/wind_dataset

Files:
- data (1).csv


In [ ]:
import pandas as pd
from pathlib import Path

csv_path = Path("/content/wind_dataset/data (1).csv")

sample_df = pd.read_csv(csv_path, nrows=5)

print("Shape:", sample_df.shape)

print("\nColumns:")
print(sample_df.columns.tolist())

print("\nDtypes:")
print(sample_df.dtypes)

print("\nHead:")
print(sample_df.head())

Shape: (5, 13)

Columns:
['YEAR', 'MONTH', 'DAY', 'HOUR', 'humidity', 'temperature', 'surface_pressure', 'wind_speed', 'wind_direction', 'Longitude', 'Latitude', 'Index', 'datetime']

Dtypes:
YEAR                  int64
MONTH                 int64
DAY                   int64
HOUR                  int64
humidity            float64
temperature         float64
surface_pressure    float64
wind_speed          float64
wind_direction      float64
Longitude           float64
Latitude            float64
Index                 int64
datetime             object
dtype: object

Head:
   YEAR  MONTH  DAY  HOUR  humidity  temperature  surface_pressure  \
0  2013      1    1     5     17.52        26.58            100.99   
1  2013      1    1     6     17.82        26.78            101.08   
2  2013      1    1     7     17.88        27.14            101.19   
3  2013      1    1     8     17.76        27.44            101.25   
4  2013      1    1     9     17.82        27.65            101.26   

  

In [ ]:
import pandas as pd
from pathlib import Path

csv_path = Path("/content/wind_dataset/data (1).csv")

dtype_map = {
    "YEAR": "int16",
    "MONTH": "int8",
    "DAY": "int8",
    "HOUR": "int8",
    "humidity": "float32",
    "temperature": "float32",
    "surface_pressure": "float32",
    "wind_speed": "float32",
    "wind_direction": "float32",
    "Longitude": "float32",
    "Latitude": "float32",
    "Index": "int16",
}

df = pd.read_csv(
    csv_path,
    dtype=dtype_map,
    parse_dates=["datetime"]
)

print("Loaded Successfully.")
print("-" * 60)

print("Shape:", df.shape)

print("\nMemory Usage (MB):")
print(df.memory_usage(deep=True).sum() / (1024**2))

print("\nDtypes:")
print(df.dtypes)

print("\nDatetime Range:")
print("Min:", df["datetime"].min())
print("Max:", df["datetime"].max())

Loaded Successfully.
------------------------------------------------------------
Shape: (2646408, 13)

Memory Usage (MB):
108.52401351928711

Dtypes:
YEAR                         int16
MONTH                         int8
DAY                           int8
HOUR                          int8
humidity                   float32
temperature                float32
surface_pressure           float32
wind_speed                 float32
wind_direction             float32
Longitude                  float32
Latitude                   float32
Index                        int16
datetime            datetime64[ns]
dtype: object

Datetime Range:
Min: 2013-01-01 05:00:00
Max: 2022-09-28 04:00:00


In [ ]:
import pandas as pd

print("=" * 80)
print("DATASET FORENSIC AUDIT")
print("=" * 80)

# ---------------------------------------------------
# Missing values
# ---------------------------------------------------
print("\n[1] Missing Values")
print("-" * 40)

missing = df.isnull().sum()

print(missing)

# ---------------------------------------------------
# Exact duplicate rows
# ---------------------------------------------------
print("\n[2] Exact Duplicate Rows")
print("-" * 40)

dup_rows = df.duplicated().sum()

print("Duplicate Rows:", dup_rows)

# ---------------------------------------------------
# Duplicate station-time keys
# ---------------------------------------------------
print("\n[3] Duplicate (Index, datetime) Pairs")
print("-" * 40)

dup_keys = df.duplicated(subset=["Index", "datetime"]).sum()

print("Duplicate Keys:", dup_keys)

# ---------------------------------------------------
# Station counts
# ---------------------------------------------------
print("\n[4] Station Coverage")
print("-" * 40)

station_counts = df.groupby("Index").size()

print("Unique Stations:", station_counts.shape[0])

print("\nRows Per Station Summary:")
print(station_counts.describe())

# ---------------------------------------------------
# Hourly continuity
# ---------------------------------------------------
print("\n[5] Hourly Continuity Check")
print("-" * 40)

continuity_issues = []

for station_id, grp in df.groupby("Index"):

    grp = grp.sort_values("datetime")

    diffs = grp["datetime"].diff().dropna()

    invalid = diffs[diffs != pd.Timedelta(hours=1)]

    if len(invalid) > 0:
        continuity_issues.append({
            "station": station_id,
            "bad_steps": len(invalid)
        })

if len(continuity_issues) == 0:
    print("All stations have perfect hourly continuity.")
else:
    print("Continuity Issues Found:")
    print(continuity_issues)

# ---------------------------------------------------
# Datetime uniqueness
# ---------------------------------------------------
print("\n[6] Global Timestamp Count")
print("-" * 40)

print("Unique timestamps:", df["datetime"].nunique())

print("\nDone.")

DATASET FORENSIC AUDIT

[1] Missing Values
----------------------------------------
YEAR                0
MONTH               0
DAY                 0
HOUR                0
humidity            0
temperature         0
surface_pressure    0
wind_speed          0
wind_direction      0
Longitude           0
Latitude            0
Index               0
datetime            0
dtype: int64

[2] Exact Duplicate Rows
----------------------------------------
Duplicate Rows: 0

[3] Duplicate (Index, datetime) Pairs
----------------------------------------
Duplicate Keys: 0

[4] Station Coverage
----------------------------------------
Unique Stations: 31

Rows Per Station Summary:
count       31.0
mean     85368.0
std          0.0
min      85368.0
25%      85368.0
50%      85368.0
75%      85368.0
max      85368.0
dtype: float64

[5] Hourly Continuity Check
----------------------------------------
All stations have perfect hourly continuity.

[6] Global Timestamp Count
------------------------------

In [ ]:
import pandas as pd
import numpy as np

print("=" * 80)
print("DEEP STATISTICAL AUDIT")
print("=" * 80)

numeric_cols = [
    "humidity",
    "temperature",
    "surface_pressure",
    "wind_speed",
    "wind_direction"
]

# ---------------------------------------------------
# Global statistics
# ---------------------------------------------------
print("\n[1] Global Numeric Statistics")
print("-" * 60)

stats = df[numeric_cols].describe().T

stats["missing_pct"] = (
    df[numeric_cols].isnull().mean() * 100
)

print(stats)

# ---------------------------------------------------
# Zero variance check
# ---------------------------------------------------
print("\n[2] Zero / Near-Zero Variance Check")
print("-" * 60)

for col in numeric_cols:
    std = df[col].std()
    unique_vals = df[col].nunique()

    print(f"{col}")
    print(f"  std         : {std:.6f}")
    print(f"  unique_vals : {unique_vals}")
    print()

# ---------------------------------------------------
# Physical plausibility checks
# ---------------------------------------------------
print("\n[3] Physical Plausibility Checks")
print("-" * 60)

checks = {
    "humidity_negative": (df["humidity"] < 0).sum(),
    "temperature_extreme_low": (df["temperature"] < -50).sum(),
    "temperature_extreme_high": (df["temperature"] > 60).sum(),
    "pressure_nonpositive": (df["surface_pressure"] <= 0).sum(),
    "wind_speed_negative": (df["wind_speed"] < 0).sum(),
    "wind_direction_outside_range": (
        ((df["wind_direction"] < 0) |
         (df["wind_direction"] > 360)).sum()
    )
}

for k, v in checks.items():
    print(f"{k}: {v}")

# ---------------------------------------------------
# Exact duplicate station signatures
# ---------------------------------------------------
print("\n[4] Station Signature Duplication")
print("-" * 60)

signature_cols = [
    "humidity",
    "temperature",
    "surface_pressure",
    "wind_speed",
    "wind_direction"
]

station_signatures = {}

for station_id, grp in df.groupby("Index"):

    sig = tuple(
        grp[signature_cols]
        .head(500)
        .round(4)
        .to_numpy()
        .flatten()
    )

    station_signatures[station_id] = sig

groups = {}

for station_id, sig in station_signatures.items():
    groups.setdefault(sig, []).append(station_id)

duplicate_groups = [
    v for v in groups.values() if len(v) > 1
]

print("Duplicate Signature Groups:")
print(duplicate_groups)

print("\nTotal Duplicate Groups:", len(duplicate_groups))

# ---------------------------------------------------
# Constant-coordinate validation
# ---------------------------------------------------
print("\n[5] Coordinate Consistency")
print("-" * 60)

coord_issues = []

for station_id, grp in df.groupby("Index"):

    lon_unique = grp["Longitude"].nunique()
    lat_unique = grp["Latitude"].nunique()

    if lon_unique != 1 or lat_unique != 1:
        coord_issues.append(station_id)

if len(coord_issues) == 0:
    print("All stations have fixed coordinates.")
else:
    print("Coordinate issues:", coord_issues)

# ---------------------------------------------------
# Datetime monotonicity
# ---------------------------------------------------
print("\n[6] Global Datetime Monotonicity")
print("-" * 60)

sorted_check = df.sort_values(
    ["Index", "datetime"]
)["datetime"].is_monotonic_increasing

print("Globally monotonic:", sorted_check)

print("\nAudit Complete.")

DEEP STATISTICAL AUDIT

[1] Global Numeric Statistics
------------------------------------------------------------
                      count        mean        std        min         25%  \
humidity          2646408.0   18.281376   1.954780   3.850000   17.209999   
temperature       2646408.0   28.125948   2.198326  12.520000   26.969999   
surface_pressure  2646408.0  100.564621   2.047544  92.160004  100.580002   
wind_speed        2646408.0    6.752097   2.704362   0.000000    4.800000   
wind_direction    2646408.0  160.890488  91.184700  -0.000000   60.380001   

                         50%         75%         max  missing_pct  
humidity           18.620001   19.650000   24.350000          0.0  
temperature        28.129999   29.400000   39.549999          0.0  
surface_pressure  100.800003  101.029999  101.930000          0.0  
wind_speed          6.880000    8.670000   23.549999          0.0  
wind_direction    197.169998  238.679993  359.959991          0.0  

[2] Zero / Ne

In [ ]:
import pandas as pd
import numpy as np

print("=" * 80)
print("TEMPORAL PREDICTABILITY AUDIT")
print("=" * 80)

# ---------------------------------------------------
# Choose representative station
# ---------------------------------------------------
station_id = 0

station_df = (
    df[df["Index"] == station_id]
    .sort_values("datetime")
    .reset_index(drop=True)
)

print(f"\nUsing Station: {station_id}")

# ---------------------------------------------------
# Variables to analyze
# ---------------------------------------------------
target_cols = [
    "temperature",
    "humidity",
    "surface_pressure",
    "wind_speed"
]

lags = [1, 6, 12, 24, 48, 72, 168]

# ---------------------------------------------------
# Autocorrelation analysis
# ---------------------------------------------------
print("\n[1] Autocorrelation Analysis")
print("-" * 60)

for col in target_cols:

    print(f"\n{col}")

    for lag in lags:

        ac = station_df[col].autocorr(lag=lag)

        print(f"  lag_{lag:>3}h : {ac:.6f}")

# ---------------------------------------------------
# Daily seasonality strength
# ---------------------------------------------------
print("\n[2] Daily Seasonality Strength")
print("-" * 60)

for col in target_cols:

    daily_corr = station_df[col].autocorr(lag=24)

    weekly_corr = station_df[col].autocorr(lag=24*7)

    print(f"\n{col}")
    print(f"  daily_corr  : {daily_corr:.6f}")
    print(f"  weekly_corr : {weekly_corr:.6f}")

# ---------------------------------------------------
# Yearly drift analysis
# ---------------------------------------------------
print("\n[3] Yearly Mean Drift")
print("-" * 60)

station_df["year"] = station_df["datetime"].dt.year

yearly_stats = (
    station_df
    .groupby("year")[target_cols]
    .mean()
)

print(yearly_stats)

# ---------------------------------------------------
# Basic persistence baseline
# ---------------------------------------------------
print("\n[4] Persistence Baseline MAE")
print("-" * 60)

horizons = [1, 6, 12, 24, 48]

for col in target_cols:

    print(f"\n{col}")

    values = station_df[col].values

    for h in horizons:

        actual = values[h:]
        pred = values[:-h]

        mae = np.mean(np.abs(actual - pred))

        print(f"  horizon_{h:>2}h MAE : {mae:.6f}")

print("\nAudit Complete.")

TEMPORAL PREDICTABILITY AUDIT

Using Station: 0

[1] Autocorrelation Analysis
------------------------------------------------------------

temperature
  lag_  1h : 0.989499
  lag_  6h : 0.745363
  lag_ 12h : 0.552127
  lag_ 24h : 0.951739
  lag_ 48h : 0.929084
  lag_ 72h : 0.913254
  lag_168h : 0.885124

humidity
  lag_  1h : 0.992482
  lag_  6h : 0.856726
  lag_ 12h : 0.747415
  lag_ 24h : 0.851054
  lag_ 48h : 0.759549
  lag_ 72h : 0.705169
  lag_168h : 0.636597

surface_pressure
  lag_  1h : 0.975508
  lag_  6h : 0.628701
  lag_ 12h : 0.925301
  lag_ 24h : 0.944964
  lag_ 48h : 0.880752
  lag_ 72h : 0.846110
  lag_168h : 0.771164

wind_speed
  lag_  1h : 0.988331
  lag_  6h : 0.748539
  lag_ 12h : 0.547807
  lag_ 24h : 0.762732
  lag_ 48h : 0.610772
  lag_ 72h : 0.494030
  lag_168h : 0.326484

[2] Daily Seasonality Strength
------------------------------------------------------------

temperature
  daily_corr  : 0.951739
  weekly_corr : 0.885124

humidity
  daily_corr  : 0.851054
 

In [ ]:
import numpy as np
import pandas as pd

print("=" * 80)
print("CANONICAL PREPROCESSING")
print("=" * 80)

# ---------------------------------------------------
# Sort deterministically
# ---------------------------------------------------
df = df.sort_values(
    ["Index", "datetime"]
).reset_index(drop=True)

print("\n[1] Sorting Complete")

# ---------------------------------------------------
# Time-based cyclical features
# ---------------------------------------------------
print("\n[2] Creating Cyclical Time Features")

# Hour
df["hour_sin"] = np.sin(2 * np.pi * df["HOUR"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["HOUR"] / 24)

# Day of week
dayofweek = df["datetime"].dt.dayofweek

df["dow_sin"] = np.sin(2 * np.pi * dayofweek / 7)
df["dow_cos"] = np.cos(2 * np.pi * dayofweek / 7)

# Month
df["month_sin"] = np.sin(2 * np.pi * df["MONTH"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["MONTH"] / 12)

print("Cyclical features created.")

# ---------------------------------------------------
# Wind direction encoding
# ---------------------------------------------------
print("\n[3] Encoding Wind Direction")

# Convert degrees -> radians
theta = np.deg2rad(df["wind_direction"])

df["dir_sin"] = np.sin(theta)
df["dir_cos"] = np.cos(theta)

print("Direction encoding complete.")

# ---------------------------------------------------
# Wind vector decomposition
# ---------------------------------------------------
print("\n[4] Creating Wind Vector Components")

df["wind_x"] = df["wind_speed"] * df["dir_cos"]
df["wind_y"] = df["wind_speed"] * df["dir_sin"]

print("Wind vectors created.")

# ---------------------------------------------------
# NaN / Inf validation
# ---------------------------------------------------
print("\n[5] Validation Checks")

new_cols = [
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "month_sin",
    "month_cos",
    "dir_sin",
    "dir_cos",
    "wind_x",
    "wind_y"
]

nan_summary = df[new_cols].isnull().sum()

print("\nNaN Counts:")
print(nan_summary)

inf_count = np.isinf(df[new_cols].to_numpy()).sum()

print("\nInf Count:", inf_count)

# ---------------------------------------------------
# Preview
# ---------------------------------------------------
print("\n[6] Preview")

preview_cols = [
    "datetime",
    "wind_speed",
    "wind_direction",
    "dir_sin",
    "dir_cos",
    "wind_x",
    "wind_y"
]

print(df[preview_cols].head())

print("\nDone.")

CANONICAL PREPROCESSING

[1] Sorting Complete

[2] Creating Cyclical Time Features
Cyclical features created.

[3] Encoding Wind Direction
Direction encoding complete.

[4] Creating Wind Vector Components
Wind vectors created.

[5] Validation Checks

NaN Counts:
hour_sin     0
hour_cos     0
dow_sin      0
dow_cos      0
month_sin    0
month_cos    0
dir_sin      0
dir_cos      0
wind_x       0
wind_y       0
dtype: int64

Inf Count: 0

[6] Preview
             datetime  wind_speed  wind_direction   dir_sin   dir_cos  \
0 2013-01-01 05:00:00        5.95       56.889999  0.837623  0.546248   
1 2013-01-01 06:00:00        5.76       55.660000  0.825705  0.564103   
2 2013-01-01 07:00:00        5.73       53.930000  0.808298  0.588773   
3 2013-01-01 08:00:00        6.02       52.799999  0.796530  0.604599   
4 2013-01-01 09:00:00        6.08       51.099998  0.778243  0.627963   

     wind_x    wind_y  
0  3.250177  4.983859  
1  3.249231  4.756059  
2  3.373670  4.631549  
3  3.639687 

In [ ]:
import pandas as pd
import numpy as np

print("=" * 80)
print("FEATURE ENGINEERING — LAGS + TARGETS")
print("=" * 80)

# ---------------------------------------------------
# Sort again defensively
# ---------------------------------------------------
df = df.sort_values(
    ["Index", "datetime"]
).reset_index(drop=True)

# ---------------------------------------------------
# Lag configuration
# ---------------------------------------------------
lag_hours = [1, 2, 3, 6, 12, 24, 48, 72, 168]

# ---------------------------------------------------
# Forecast horizons
# ---------------------------------------------------
horizons = [1, 6, 12, 24, 48]

# ---------------------------------------------------
# Base variable for lags
# ---------------------------------------------------
base_col = "wind_speed"

# ---------------------------------------------------
# Create lag features
# ---------------------------------------------------
print("\n[1] Creating Lag Features")

for lag in lag_hours:

    col_name = f"{base_col}_lag_{lag}"

    df[col_name] = (
        df.groupby("Index")[base_col]
        .shift(lag)
        .astype("float32")
    )

print(f"Created {len(lag_hours)} lag features.")

# ---------------------------------------------------
# Rolling statistics
# ---------------------------------------------------
print("\n[2] Creating Rolling Features")

rolling_windows = [6, 12, 24, 48]

for window in rolling_windows:

    mean_col = f"{base_col}_roll_mean_{window}"
    std_col = f"{base_col}_roll_std_{window}"

    grouped = df.groupby("Index")[base_col]

    df[mean_col] = (
        grouped
        .rolling(window)
        .mean()
        .reset_index(level=0, drop=True)
        .astype("float32")
    )

    df[std_col] = (
        grouped
        .rolling(window)
        .std()
        .reset_index(level=0, drop=True)
        .astype("float32")
    )

print(f"Created {len(rolling_windows)*2} rolling features.")

# ---------------------------------------------------
# Multi-horizon targets
# ---------------------------------------------------
print("\n[3] Creating Forecast Targets")

for h in horizons:

    target_col = f"target_t_plus_{h}"

    df[target_col] = (
        df.groupby("Index")[base_col]
        .shift(-h)
        .astype("float32")
    )

print(f"Created {len(horizons)} target columns.")

# ---------------------------------------------------
# NaN summary
# ---------------------------------------------------
print("\n[4] NaN Summary")

engineered_cols = [
    c for c in df.columns
    if (
        "lag_" in c or
        "roll_" in c or
        "target_t_plus_" in c
    )
]

nan_summary = (
    df[engineered_cols]
    .isnull()
    .sum()
    .sort_values(ascending=False)
)

print(nan_summary.head(20))

# ---------------------------------------------------
# Memory usage
# ---------------------------------------------------
print("\n[5] Memory Usage")

mem_mb = df.memory_usage(deep=True).sum() / (1024**2)

print(f"Current DataFrame Memory: {mem_mb:.2f} MB")

# ---------------------------------------------------
# Preview
# ---------------------------------------------------
print("\n[6] Preview")

preview_cols = [
    "datetime",
    "Index",
    "wind_speed",
    "wind_speed_lag_1",
    "wind_speed_lag_24",
    "wind_speed_roll_mean_24",
    "target_t_plus_1",
    "target_t_plus_24"
]

print(df[preview_cols].head(10))

print("\nDone.")

FEATURE ENGINEERING — LAGS + TARGETS

[1] Creating Lag Features
Created 9 lag features.

[2] Creating Rolling Features
Created 8 rolling features.

[3] Creating Forecast Targets
Created 5 target columns.

[4] NaN Summary
wind_speed_lag_168         5208
wind_speed_lag_72          2232
target_t_plus_48           1488
wind_speed_lag_48          1488
wind_speed_roll_std_48     1457
wind_speed_roll_mean_48    1457
target_t_plus_24            744
wind_speed_lag_24           744
wind_speed_roll_std_24      713
wind_speed_roll_mean_24     713
wind_speed_lag_12           372
target_t_plus_12            372
wind_speed_roll_mean_12     341
wind_speed_roll_std_12      341
target_t_plus_6             186
wind_speed_lag_6            186
wind_speed_roll_std_6       155
wind_speed_roll_mean_6      155
wind_speed_lag_3             93
wind_speed_lag_2             62
dtype: int64

[5] Memory Usage
Current DataFrame Memory: 492.14 MB

[6] Preview
             datetime  Index  wind_speed  wind_speed_lag_1 

In [ ]:
import pandas as pd
import numpy as np

print("=" * 80)
print("MODELING DATASET PREPARATION")
print("=" * 80)

# ---------------------------------------------------
# Target horizon
# ---------------------------------------------------
TARGET_HORIZON = 1

target_col = f"target_t_plus_{TARGET_HORIZON}"

print(f"\nTarget Column: {target_col}")

# ---------------------------------------------------
# Feature columns
# ---------------------------------------------------
feature_cols = [
    # Wind history
    "wind_speed_lag_1",
    "wind_speed_lag_2",
    "wind_speed_lag_3",
    "wind_speed_lag_6",
    "wind_speed_lag_12",
    "wind_speed_lag_24",
    "wind_speed_lag_48",
    "wind_speed_lag_72",
    "wind_speed_lag_168",

    # Rolling stats
    "wind_speed_roll_mean_6",
    "wind_speed_roll_std_6",
    "wind_speed_roll_mean_12",
    "wind_speed_roll_std_12",
    "wind_speed_roll_mean_24",
    "wind_speed_roll_std_24",
    "wind_speed_roll_mean_48",
    "wind_speed_roll_std_48",

    # Direction encoding
    "dir_sin",
    "dir_cos",

    # Wind vectors
    "wind_x",
    "wind_y",

    # Cyclical time features
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "month_sin",
    "month_cos",

    # Station metadata
    "Longitude",
    "Latitude",
    "Index"
]

print(f"\nNumber of Features: {len(feature_cols)}")

# ---------------------------------------------------
# Keep only required columns
# ---------------------------------------------------
model_df = df[
    ["datetime"] + feature_cols + [target_col]
].copy()

# ---------------------------------------------------
# Drop NaNs safely
# ---------------------------------------------------
before_rows = len(model_df)

model_df = model_df.dropna().reset_index(drop=True)

after_rows = len(model_df)

print("\n[1] NaN Removal")
print("-" * 40)

print(f"Rows Before: {before_rows:,}")
print(f"Rows After : {after_rows:,}")
print(f"Rows Removed: {before_rows - after_rows:,}")

# ---------------------------------------------------
# Chronological split
# ---------------------------------------------------
print("\n[2] Chronological Split")
print("-" * 40)

train_df = model_df[
    model_df["datetime"] < "2021-01-01"
]

valid_df = model_df[
    (model_df["datetime"] >= "2021-01-01") &
    (model_df["datetime"] < "2022-01-01")
]

test_df = model_df[
    model_df["datetime"] >= "2022-01-01"
]

print(f"Train Rows      : {len(train_df):,}")
print(f"Validation Rows : {len(valid_df):,}")
print(f"Test Rows       : {len(test_df):,}")

# ---------------------------------------------------
# Final feature matrices
# ---------------------------------------------------
X_train = train_df[feature_cols]
y_train = train_df[target_col]

X_valid = valid_df[feature_cols]
y_valid = valid_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

# ---------------------------------------------------
# Validation checks
# ---------------------------------------------------
print("\n[3] Validation Checks")
print("-" * 40)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("\nAny NaNs Remaining?")
print("Train:", X_train.isnull().sum().sum())
print("Valid:", X_valid.isnull().sum().sum())
print("Test :", X_test.isnull().sum().sum())

print("\nTarget Statistics")
print(y_train.describe())

print("\nDone.")

MODELING DATASET PREPARATION

Target Column: target_t_plus_1

Number of Features: 30

[1] NaN Removal
----------------------------------------
Rows Before: 2,646,408
Rows After : 2,641,169
Rows Removed: 5,239

[2] Chronological Split
----------------------------------------
Train Rows      : 2,168,605
Validation Rows : 271,560
Test Rows       : 201,004

[3] Validation Checks
----------------------------------------
X_train shape: (2168605, 30)
y_train shape: (2168605,)

Any NaNs Remaining?
Train: 0
Valid: 0
Test : 0

Target Statistics
count    2.168605e+06
mean     6.766513e+00
std      2.706131e+00
min      0.000000e+00
25%      4.810000e+00
50%      6.900000e+00
75%      8.690000e+00
max      2.355000e+01
Name: target_t_plus_1, dtype: float64

Done.


In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import numpy as np
import pandas as pd

print("=" * 80)
print("BASELINE EVALUATION")
print("=" * 80)

# ---------------------------------------------------
# Actual values
# ---------------------------------------------------
y_true = y_valid.values

# ---------------------------------------------------
# Baseline A: Persistence
# ---------------------------------------------------
print("\n[1] Persistence Baseline")
print("-" * 60)

pred_persistence = X_valid["wind_speed_lag_1"].values

mae_p = mean_absolute_error(y_true, pred_persistence)

rmse_p = np.sqrt(
    mean_squared_error(y_true, pred_persistence)
)

r2_p = r2_score(y_true, pred_persistence)

print(f"MAE  : {mae_p:.6f}")
print(f"RMSE : {rmse_p:.6f}")
print(f"R²   : {r2_p:.6f}")

# ---------------------------------------------------
# Baseline B: Daily seasonal persistence
# ---------------------------------------------------
print("\n[2] Daily Seasonal Baseline")
print("-" * 60)

pred_daily = X_valid["wind_speed_lag_24"].values

mae_d = mean_absolute_error(y_true, pred_daily)

rmse_d = np.sqrt(
    mean_squared_error(y_true, pred_daily)
)

r2_d = r2_score(y_true, pred_daily)

print(f"MAE  : {mae_d:.6f}")
print(f"RMSE : {rmse_d:.6f}")
print(f"R²   : {r2_d:.6f}")

# ---------------------------------------------------
# Improvement comparison
# ---------------------------------------------------
print("\n[3] Baseline Comparison")
print("-" * 60)

if mae_d < mae_p:
    better = "Daily Seasonal Baseline"
else:
    better = "Persistence Baseline"

print(f"Better Baseline: {better}")

print("\nMAE Difference:")
print(f"{abs(mae_d - mae_p):.6f}")

print("\nDone.")

BASELINE EVALUATION

[1] Persistence Baseline
------------------------------------------------------------
MAE  : 0.539816
RMSE : 0.700740
R²   : 0.931288

[2] Daily Seasonal Baseline
------------------------------------------------------------
MAE  : 1.451683
RMSE : 1.909987
R²   : 0.489522

[3] Baseline Comparison
------------------------------------------------------------
Better Baseline: Persistence Baseline

MAE Difference:
0.911867

Done.


In [ ]:
# Install LightGBM if needed
!pip -q install lightgbm

In [ ]:
import lightgbm as lgb

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import numpy as np
import pandas as pd
import time

print("=" * 80)
print("LIGHTGBM BASELINE TRAINING")
print("=" * 80)

# ---------------------------------------------------
# Model configuration
# ---------------------------------------------------
model = lgb.LGBMRegressor(
    objective="regression",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=64,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

# ---------------------------------------------------
# Train
# ---------------------------------------------------
print("\n[1] Training Model")

start_time = time.time()

model.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)],
    eval_metric="l1"
)

train_time = time.time() - start_time

print(f"\nTraining Time: {train_time:.2f} seconds")

# ---------------------------------------------------
# Predict
# ---------------------------------------------------
print("\n[2] Generating Predictions")

preds = model.predict(X_valid)

# ---------------------------------------------------
# Metrics
# ---------------------------------------------------
print("\n[3] Validation Metrics")
print("-" * 60)

mae = mean_absolute_error(y_valid, preds)

rmse = np.sqrt(
    mean_squared_error(y_valid, preds)
)

r2 = r2_score(y_valid, preds)

print(f"LightGBM MAE  : {mae:.6f}")
print(f"LightGBM RMSE : {rmse:.6f}")
print(f"LightGBM R²   : {r2:.6f}")

# ---------------------------------------------------
# Compare vs persistence
# ---------------------------------------------------
print("\n[4] Improvement vs Persistence")
print("-" * 60)

baseline_mae = 0.539816
baseline_rmse = 0.700740
baseline_r2 = 0.931288

print(f"MAE Improvement  : {baseline_mae - mae:.6f}")
print(f"RMSE Improvement : {baseline_rmse - rmse:.6f}")
print(f"R² Improvement   : {r2 - baseline_r2:.6f}")

# ---------------------------------------------------
# Feature importance
# ---------------------------------------------------
print("\n[5] Top Feature Importances")
print("-" * 60)

importance_df = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model.feature_importances_
})

importance_df = (
    importance_df
    .sort_values("importance", ascending=False)
)

print(importance_df.head(15))

print("\nDone.")

LIGHTGBM BASELINE TRAINING

[1] Training Model
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.862934 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5485
[LightGBM] [Info] Number of data points in the train set: 2168605, number of used features: 30
[LightGBM] [Info] Start training from score 6.766514

Training Time: 142.23 seconds

[2] Generating Predictions

[3] Validation Metrics
------------------------------------------------------------
LightGBM MAE  : 0.235299
LightGBM RMSE : 0.319085
LightGBM R²   : 0.985753

[4] Improvement vs Persistence
------------------------------------------------------------
MAE Improvement  : 0.304517
RMSE Improvement : 0.381655
R² Improvement   : 0.054465

[5] Top Feature Importances
------------------------------------------------------------
                    feature  importance
20                   wind_y        3253
1          wind_speed_lag_2        2767
19 

In [ ]:
import lightgbm as lgb

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import numpy as np
import pandas as pd
import time

print("=" * 80)
print("MULTI-HORIZON FORECASTING BENCHMARK")
print("=" * 80)

# ---------------------------------------------------
# Forecast horizons
# ---------------------------------------------------
horizons = [1, 6, 12, 24, 48]

results = []

# ---------------------------------------------------
# Training loop
# ---------------------------------------------------
for horizon in horizons:

    print(f"\n{'='*60}")
    print(f"HORIZON: t+{horizon}h")
    print(f"{'='*60}")

    target_col = f"target_t_plus_{horizon}"

    # -----------------------------------------------
    # Build temporary modeling dataset
    # -----------------------------------------------
    temp_df = df[
        ["datetime"] + feature_cols + [target_col]
    ].dropna()

    train_df_h = temp_df[
        temp_df["datetime"] < "2021-01-01"
    ]

    valid_df_h = temp_df[
        (temp_df["datetime"] >= "2021-01-01") &
        (temp_df["datetime"] < "2022-01-01")
    ]

    X_train_h = train_df_h[feature_cols]
    y_train_h = train_df_h[target_col]

    X_valid_h = valid_df_h[feature_cols]
    y_valid_h = valid_df_h[target_col]

    # -----------------------------------------------
    # Model
    # -----------------------------------------------
    model_h = lgb.LGBMRegressor(
        objective="regression",
        n_estimators=200,
        learning_rate=0.05,
        num_leaves=64,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )

    # -----------------------------------------------
    # Train
    # -----------------------------------------------
    start = time.time()

    model_h.fit(
        X_train_h,
        y_train_h
    )

    train_time = time.time() - start

    # -----------------------------------------------
    # Predict
    # -----------------------------------------------
    preds = model_h.predict(X_valid_h)

    # -----------------------------------------------
    # Metrics
    # -----------------------------------------------
    mae = mean_absolute_error(y_valid_h, preds)

    rmse = np.sqrt(
        mean_squared_error(y_valid_h, preds)
    )

    r2 = r2_score(y_valid_h, preds)

    print(f"MAE  : {mae:.6f}")
    print(f"RMSE : {rmse:.6f}")
    print(f"R²   : {r2:.6f}")
    print(f"Train Time (s): {train_time:.2f}")

    results.append({
        "horizon": horizon,
        "mae": mae,
        "rmse": rmse,
        "r2": r2,
        "train_time_sec": train_time
    })

# ---------------------------------------------------
# Final summary table
# ---------------------------------------------------
print("\n" + "="*80)
print("FINAL RESULTS")
print("="*80)

results_df = pd.DataFrame(results)

print(results_df)

print("\nDone.")

MULTI-HORIZON FORECASTING BENCHMARK

HORIZON: t+1h
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.266465 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5485
[LightGBM] [Info] Number of data points in the train set: 2168605, number of used features: 30
[LightGBM] [Info] Start training from score 6.766514
MAE  : 0.254491
RMSE : 0.344207
R²   : 0.983421
Train Time (s): 92.50

HORIZON: t+6h
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.688271 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5485
[LightGBM] [Info] Number of data points in the train set: 2168605, number of used features: 30
[LightGBM] [Info] Start training from score 6.766583
MAE  : 0.879718
RMSE : 1.147615
R²   : 0.815674
Train Time (s): 85.96

HORIZON: t+12h
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.71

In [ ]:
import lightgbm as lgb

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import numpy as np
import pandas as pd
import time

print("=" * 80)
print("WALK-FORWARD EXPANDING-WINDOW VALIDATION")
print("=" * 80)

# ---------------------------------------------------
# Use 1h horizon
# ---------------------------------------------------
TARGET_HORIZON = 1

target_col = f"target_t_plus_{TARGET_HORIZON}"

# ---------------------------------------------------
# Modeling dataframe
# ---------------------------------------------------
wf_df = df[
    ["datetime"] + feature_cols + [target_col]
].dropna()

# ---------------------------------------------------
# Walk-forward folds
# ---------------------------------------------------
folds = [
    ("2013-01-01", "2018-01-01"),
    ("2013-01-01", "2019-01-01"),
    ("2013-01-01", "2020-01-01"),
    ("2013-01-01", "2021-01-01"),
]

results = []

# ---------------------------------------------------
# Walk-forward loop
# ---------------------------------------------------
for i in range(len(folds) - 1):

    train_start = folds[i][0]
    train_end = folds[i][1]

    valid_start = train_end
    valid_end = folds[i + 1][1]

    print(f"\n{'='*60}")
    print(f"FOLD {i+1}")
    print(f"{'='*60}")

    print(f"Train : {train_start} → {train_end}")
    print(f"Valid : {valid_start} → {valid_end}")

    # -----------------------------------------------
    # Split
    # -----------------------------------------------
    train_df_fold = wf_df[
        (wf_df["datetime"] >= train_start) &
        (wf_df["datetime"] < train_end)
    ]

    valid_df_fold = wf_df[
        (wf_df["datetime"] >= valid_start) &
        (wf_df["datetime"] < valid_end)
    ]

    X_train_fold = train_df_fold[feature_cols]
    y_train_fold = train_df_fold[target_col]

    X_valid_fold = valid_df_fold[feature_cols]
    y_valid_fold = valid_df_fold[target_col]

    # -----------------------------------------------
    # Model
    # -----------------------------------------------
    model_fold = lgb.LGBMRegressor(
        objective="regression",
        n_estimators=200,
        learning_rate=0.05,
        num_leaves=64,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )

    # -----------------------------------------------
    # Train
    # -----------------------------------------------
    start_time = time.time()

    model_fold.fit(
        X_train_fold,
        y_train_fold
    )

    train_time = time.time() - start_time

    # -----------------------------------------------
    # Predict
    # -----------------------------------------------
    preds = model_fold.predict(X_valid_fold)

    # -----------------------------------------------
    # Metrics
    # -----------------------------------------------
    mae = mean_absolute_error(
        y_valid_fold,
        preds
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_valid_fold,
            preds
        )
    )

    r2 = r2_score(
        y_valid_fold,
        preds
    )

    print(f"\nMAE  : {mae:.6f}")
    print(f"RMSE : {rmse:.6f}")
    print(f"R²   : {r2:.6f}")
    print(f"Train Time (s): {train_time:.2f}")

    results.append({
        "fold": i + 1,
        "train_end": train_end,
        "valid_end": valid_end,
        "mae": mae,
        "rmse": rmse,
        "r2": r2,
        "train_time_sec": train_time
    })

# ---------------------------------------------------
# Final results
# ---------------------------------------------------
print("\n" + "=" * 80)
print("FINAL WALK-FORWARD RESULTS")
print("=" * 80)

results_df = pd.DataFrame(results)

print(results_df)

print("\nMetric Stability")
print("-" * 40)

print(f"MAE Mean : {results_df['mae'].mean():.6f}")
print(f"MAE Std  : {results_df['mae'].std():.6f}")

print(f"\nR² Mean  : {results_df['r2'].mean():.6f}")
print(f"R² Std   : {results_df['r2'].std():.6f}")

print("\nDone.")

WALK-FORWARD EXPANDING-WINDOW VALIDATION

FOLD 1
Train : 2013-01-01 → 2018-01-01
Valid : 2018-01-01 → 2019-01-01
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.427394 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5485
[LightGBM] [Info] Number of data points in the train set: 1353181, number of used features: 30
[LightGBM] [Info] Start training from score 6.800436

MAE  : 0.272497
RMSE : 0.365701
R²   : 0.981392
Train Time (s): 63.23

FOLD 2
Train : 2013-01-01 → 2019-01-01
Valid : 2019-01-01 → 2020-01-01
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.789114 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5485
[LightGBM] [Info] Number of data points in the train set: 1624741, number of used features: 30
[LightGBM] [Info] Start training from score 6.767471

MAE  : 0.266999
RMSE : 0.355107
R²   : 0.983570
Train T

In [ ]:
import numpy as np
import pandas as pd

print("=" * 80)
print("ADVANCED FEATURE ENGINEERING")
print("=" * 80)

# ---------------------------------------------------
# Defensive sorting
# ---------------------------------------------------
df = df.sort_values(
    ["Index", "datetime"]
).reset_index(drop=True)

grouped = df.groupby("Index")

# ---------------------------------------------------
# 1. Wind acceleration features
# ---------------------------------------------------
print("\n[1] Wind Acceleration Features")

df["wind_accel_1"] = (
    grouped["wind_speed"]
    .diff(1)
    .astype("float32")
)

df["wind_accel_3"] = (
    grouped["wind_speed"]
    .diff(3)
    .astype("float32")
)

df["wind_accel_6"] = (
    grouped["wind_speed"]
    .diff(6)
    .astype("float32")
)

print("Acceleration features created.")

# ---------------------------------------------------
# 2. Directional change dynamics
# ---------------------------------------------------
print("\n[2] Directional Dynamics")

df["dir_change_sin"] = (
    grouped["dir_sin"]
    .diff(1)
    .astype("float32")
)

df["dir_change_cos"] = (
    grouped["dir_cos"]
    .diff(1)
    .astype("float32")
)

print("Directional dynamics created.")

# ---------------------------------------------------
# 3. Exponentially weighted moving averages
# ---------------------------------------------------
print("\n[3] EWMA Features")

ewma_windows = [6, 12, 24]

for span in ewma_windows:

    col_name = f"wind_ewm_{span}"

    df[col_name] = (
        grouped["wind_speed"]
        .transform(
            lambda x: x.ewm(
                span=span,
                adjust=False
            ).mean()
        )
        .astype("float32")
    )

print("EWMA features created.")

# ---------------------------------------------------
# 4. Multi-scale volatility
# ---------------------------------------------------
print("\n[4] Multi-scale Volatility")

vol_windows = [6, 24, 72]

for w in vol_windows:

    col_name = f"wind_volatility_{w}"

    df[col_name] = (
        grouped["wind_speed"]
        .rolling(w)
        .std()
        .reset_index(level=0, drop=True)
        .astype("float32")
    )

print("Volatility features created.")

# ---------------------------------------------------
# 5. Interaction features
# ---------------------------------------------------
print("\n[5] Interaction Features")

df["speed_x_hour_sin"] = (
    df["wind_speed"] * df["hour_sin"]
).astype("float32")

df["speed_x_hour_cos"] = (
    df["wind_speed"] * df["hour_cos"]
).astype("float32")

df["speed_x_dir_sin"] = (
    df["wind_speed"] * df["dir_sin"]
).astype("float32")

df["speed_x_dir_cos"] = (
    df["wind_speed"] * df["dir_cos"]
).astype("float32")

print("Interaction features created.")

# ---------------------------------------------------
# NaN summary
# ---------------------------------------------------
print("\n[6] NaN Summary")

new_feature_cols = [
    "wind_accel_1",
    "wind_accel_3",
    "wind_accel_6",
    "dir_change_sin",
    "dir_change_cos",
    "wind_ewm_6",
    "wind_ewm_12",
    "wind_ewm_24",
    "wind_volatility_6",
    "wind_volatility_24",
    "wind_volatility_72",
    "speed_x_hour_sin",
    "speed_x_hour_cos",
    "speed_x_dir_sin",
    "speed_x_dir_cos"
]

nan_summary = (
    df[new_feature_cols]
    .isnull()
    .sum()
)

print(nan_summary)

# ---------------------------------------------------
# Memory check
# ---------------------------------------------------
print("\n[7] Memory Usage")

mem_mb = df.memory_usage(deep=True).sum() / (1024**2)

print(f"Current DataFrame Memory: {mem_mb:.2f} MB")

print("\nDone.")

ADVANCED FEATURE ENGINEERING

[1] Wind Acceleration Features
Acceleration features created.

[2] Directional Dynamics
Directional dynamics created.

[3] EWMA Features
EWMA features created.

[4] Multi-scale Volatility
Volatility features created.

[5] Interaction Features
Interaction features created.

[6] NaN Summary
wind_accel_1            31
wind_accel_3            93
wind_accel_6           186
dir_change_sin          31
dir_change_cos          31
wind_ewm_6               0
wind_ewm_12              0
wind_ewm_24              0
wind_volatility_6      155
wind_volatility_24     713
wind_volatility_72    2201
speed_x_hour_sin         0
speed_x_hour_cos         0
speed_x_dir_sin          0
speed_x_dir_cos          0
dtype: int64

[7] Memory Usage
Current DataFrame Memory: 643.57 MB

Done.


In [ ]:
import lightgbm as lgb

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import numpy as np
import pandas as pd
import time

print("=" * 80)
print("LONG-HORIZON IMPROVEMENT TEST")
print("=" * 80)

# ---------------------------------------------------
# Enhanced feature set
# ---------------------------------------------------
enhanced_features = feature_cols + [

    # Acceleration
    "wind_accel_1",
    "wind_accel_3",
    "wind_accel_6",

    # Directional dynamics
    "dir_change_sin",
    "dir_change_cos",

    # EWMA
    "wind_ewm_6",
    "wind_ewm_12",
    "wind_ewm_24",

    # Volatility
    "wind_volatility_6",
    "wind_volatility_24",
    "wind_volatility_72",

    # Interactions
    "speed_x_hour_sin",
    "speed_x_hour_cos",
    "speed_x_dir_sin",
    "speed_x_dir_cos"
]

print(f"\nTotal Features: {len(enhanced_features)}")

# ---------------------------------------------------
# Previous benchmark results
# ---------------------------------------------------
previous_results = {
    24: {
        "mae": 1.313672,
        "rmse": 1.724873,
        "r2": 0.583691
    },
    48: {
        "mae": 1.613367,
        "rmse": 2.091909,
        "r2": 0.389178
    }
}

# ---------------------------------------------------
# Horizons to test
# ---------------------------------------------------
test_horizons = [24, 48]

results = []

# ---------------------------------------------------
# Training loop
# ---------------------------------------------------
for horizon in test_horizons:

    print(f"\n{'='*60}")
    print(f"HORIZON: t+{horizon}h")
    print(f"{'='*60}")

    target_col = f"target_t_plus_{horizon}"

    temp_df = df[
        ["datetime"] + enhanced_features + [target_col]
    ].dropna()

    # -----------------------------------------------
    # Chronological split
    # -----------------------------------------------
    train_df_h = temp_df[
        temp_df["datetime"] < "2021-01-01"
    ]

    valid_df_h = temp_df[
        (temp_df["datetime"] >= "2021-01-01") &
        (temp_df["datetime"] < "2022-01-01")
    ]

    X_train_h = train_df_h[enhanced_features]
    y_train_h = train_df_h[target_col]

    X_valid_h = valid_df_h[enhanced_features]
    y_valid_h = valid_df_h[target_col]

    # -----------------------------------------------
    # Model
    # -----------------------------------------------
    model_h = lgb.LGBMRegressor(
        objective="regression",
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=96,
        max_depth=10,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )

    # -----------------------------------------------
    # Train
    # -----------------------------------------------
    start = time.time()

    model_h.fit(
        X_train_h,
        y_train_h
    )

    train_time = time.time() - start

    # -----------------------------------------------
    # Predict
    # -----------------------------------------------
    preds = model_h.predict(X_valid_h)

    # -----------------------------------------------
    # Metrics
    # -----------------------------------------------
    mae = mean_absolute_error(
        y_valid_h,
        preds
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_valid_h,
            preds
        )
    )

    r2 = r2_score(
        y_valid_h,
        preds
    )

    old = previous_results[horizon]

    print(f"\nNEW RESULTS")
    print(f"MAE  : {mae:.6f}")
    print(f"RMSE : {rmse:.6f}")
    print(f"R²   : {r2:.6f}")

    print(f"\nIMPROVEMENT VS OLD")
    print(f"MAE Gain  : {old['mae'] - mae:.6f}")
    print(f"RMSE Gain : {old['rmse'] - rmse:.6f}")
    print(f"R² Gain   : {r2 - old['r2']:.6f}")

    print(f"\nTrain Time: {train_time:.2f} sec")

    results.append({
        "horizon": horizon,
        "old_r2": old["r2"],
        "new_r2": r2,
        "r2_gain": r2 - old["r2"],
        "old_mae": old["mae"],
        "new_mae": mae,
        "mae_gain": old["mae"] - mae
    })

# ---------------------------------------------------
# Final summary
# ---------------------------------------------------
print("\n" + "="*80)
print("FINAL COMPARISON")
print("="*80)

results_df = pd.DataFrame(results)

print(results_df)

print("\nDone.")

LONG-HORIZON IMPROVEMENT TEST

Total Features: 45

HORIZON: t+24h
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.722983 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 9310
[LightGBM] [Info] Number of data points in the train set: 2168605, number of used features: 45
[LightGBM] [Info] Start training from score 6.766858

NEW RESULTS
MAE  : 1.294034
RMSE : 1.698896
R²   : 0.596136

IMPROVEMENT VS OLD
MAE Gain  : 0.019638
RMSE Gain : 0.025977
R² Gain   : 0.012445

Train Time: 172.47 sec

HORIZON: t+48h
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.085857 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 9310
[LightGBM] [Info] Number of data points in the train set: 2168605, number of used features: 45
[LightGBM] [Info] Start training from score 6.766602

NEW RESULTS
MAE  : 1.597361
RMSE : 2.073863
R²   : 0.399671



In [ ]:
import pandas as pd
import numpy as np

print("=" * 80)
print("SPATIAL-TEMPORAL FEATURE ENGINEERING")
print("=" * 80)

# ---------------------------------------------------
# Regional hourly aggregates
# ---------------------------------------------------
print("\n[1] Regional Atmospheric Context")

regional = (
    df.groupby("datetime")
    .agg({
        "wind_speed": ["mean", "std"],
        "surface_pressure": ["mean", "std"],
        "humidity": ["mean", "std"]
    })
)

regional.columns = [
    "regional_ws_mean",
    "regional_ws_std",
    "regional_pressure_mean",
    "regional_pressure_std",
    "regional_humidity_mean",
    "regional_humidity_std"
]

regional = regional.reset_index()

print("Regional aggregates created.")

# ---------------------------------------------------
# Merge back
# ---------------------------------------------------
print("\n[2] Merging Regional Context")

df = df.merge(
    regional,
    on="datetime",
    how="left"
)

print("Merge complete.")

# ---------------------------------------------------
# Relative-state features
# ---------------------------------------------------
print("\n[3] Relative-State Features")

df["ws_vs_region"] = (
    df["wind_speed"] -
    df["regional_ws_mean"]
).astype("float32")

df["pressure_vs_region"] = (
    df["surface_pressure"] -
    df["regional_pressure_mean"]
).astype("float32")

df["humidity_vs_region"] = (
    df["humidity"] -
    df["regional_humidity_mean"]
).astype("float32")

print("Relative-state features created.")

# ---------------------------------------------------
# Normalized anomaly features
# ---------------------------------------------------
print("\n[4] Normalized Atmospheric Anomalies")

EPS = 1e-6

df["ws_anomaly"] = (
    (
        df["wind_speed"] -
        df["regional_ws_mean"]
    ) /
    (df["regional_ws_std"] + EPS)
).astype("float32")

df["pressure_anomaly"] = (
    (
        df["surface_pressure"] -
        df["regional_pressure_mean"]
    ) /
    (df["regional_pressure_std"] + EPS)
).astype("float32")

print("Anomaly features created.")

# ---------------------------------------------------
# NaN validation
# ---------------------------------------------------
print("\n[5] NaN Validation")

spatial_cols = [
    "regional_ws_mean",
    "regional_ws_std",
    "regional_pressure_mean",
    "regional_pressure_std",
    "regional_humidity_mean",
    "regional_humidity_std",
    "ws_vs_region",
    "pressure_vs_region",
    "humidity_vs_region",
    "ws_anomaly",
    "pressure_anomaly"
]

print(
    df[spatial_cols]
    .isnull()
    .sum()
)

# ---------------------------------------------------
# Memory usage
# ---------------------------------------------------
print("\n[6] Memory Usage")

mem_mb = (
    df.memory_usage(deep=True)
    .sum() / (1024**2)
)

print(f"Current Memory: {mem_mb:.2f} MB")

print("\nDone.")

SPATIAL-TEMPORAL FEATURE ENGINEERING

[1] Regional Atmospheric Context
Regional aggregates created.

[2] Merging Regional Context
Merge complete.

[3] Relative-State Features
Relative-state features created.

[4] Normalized Atmospheric Anomalies
Anomaly features created.

[5] NaN Validation
regional_ws_mean          0
regional_ws_std           0
regional_pressure_mean    0
regional_pressure_std     0
regional_humidity_mean    0
regional_humidity_std     0
ws_vs_region              0
pressure_vs_region        0
humidity_vs_region        0
ws_anomaly                0
pressure_anomaly          0
dtype: int64

[6] Memory Usage
Current Memory: 754.62 MB

Done.


In [ ]:
import lightgbm as lgb

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import numpy as np
import pandas as pd
import time

print("=" * 80)
print("SPATIAL-CONTEXT 24H FORECAST TEST")
print("=" * 80)

# ---------------------------------------------------
# Enhanced + spatial feature set
# ---------------------------------------------------
spatial_features = enhanced_features + [

    # Regional atmospheric state
    "regional_ws_mean",
    "regional_ws_std",

    "regional_pressure_mean",
    "regional_pressure_std",

    "regional_humidity_mean",
    "regional_humidity_std",

    # Relative-state features
    "ws_vs_region",
    "pressure_vs_region",
    "humidity_vs_region",

    # Normalized anomalies
    "ws_anomaly",
    "pressure_anomaly"
]

print(f"\nTotal Features: {len(spatial_features)}")

# ---------------------------------------------------
# Target
# ---------------------------------------------------
TARGET_HORIZON = 24

target_col = f"target_t_plus_{TARGET_HORIZON}"

# ---------------------------------------------------
# Dataset
# ---------------------------------------------------
temp_df = df[
    ["datetime"] + spatial_features + [target_col]
].dropna()

# ---------------------------------------------------
# Chronological split
# ---------------------------------------------------
train_df = temp_df[
    temp_df["datetime"] < "2021-01-01"
]

valid_df = temp_df[
    (temp_df["datetime"] >= "2021-01-01") &
    (temp_df["datetime"] < "2022-01-01")
]

X_train = train_df[spatial_features]
y_train = train_df[target_col]

X_valid = valid_df[spatial_features]
y_valid = valid_df[target_col]

# ---------------------------------------------------
# Model
# ---------------------------------------------------
model = lgb.LGBMRegressor(
    objective="regression",
    n_estimators=350,
    learning_rate=0.04,
    num_leaves=128,
    max_depth=12,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

# ---------------------------------------------------
# Train
# ---------------------------------------------------
print("\n[1] Training")

start = time.time()

model.fit(
    X_train,
    y_train
)

train_time = time.time() - start

print(f"Training Time: {train_time:.2f} sec")

# ---------------------------------------------------
# Predict
# ---------------------------------------------------
print("\n[2] Predicting")

preds = model.predict(X_valid)

# ---------------------------------------------------
# Metrics
# ---------------------------------------------------
print("\n[3] Metrics")
print("-" * 60)

mae = mean_absolute_error(
    y_valid,
    preds
)

rmse = np.sqrt(
    mean_squared_error(
        y_valid,
        preds
    )
)

r2 = r2_score(
    y_valid,
    preds
)

print(f"MAE  : {mae:.6f}")
print(f"RMSE : {rmse:.6f}")
print(f"R²   : {r2:.6f}")

# ---------------------------------------------------
# Historical comparison
# ---------------------------------------------------
print("\n[4] Comparison")
print("-" * 60)

baseline_r2 = 0.583691
advanced_r2 = 0.596136

baseline_mae = 1.313672
advanced_mae = 1.294034

print("vs Original Baseline")
print(f"R² Gain  : {r2 - baseline_r2:.6f}")
print(f"MAE Gain : {baseline_mae - mae:.6f}")

print("\nvs Advanced Dynamic Features")
print(f"R² Gain  : {r2 - advanced_r2:.6f}")
print(f"MAE Gain : {advanced_mae - mae:.6f}")

# ---------------------------------------------------
# Feature importance
# ---------------------------------------------------
print("\n[5] Top Features")
print("-" * 60)

importance_df = pd.DataFrame({
    "feature": spatial_features,
    "importance": model.feature_importances_
})

importance_df = (
    importance_df
    .sort_values("importance", ascending=False)
)

print(importance_df.head(20))

print("\nDone.")

SPATIAL-CONTEXT 24H FORECAST TEST

Total Features: 56

[1] Training
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 2.491346 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12115
[LightGBM] [Info] Number of data points in the train set: 2168605, number of used features: 56
[LightGBM] [Info] Start training from score 6.766858
Training Time: 278.53 sec

[2] Predicting

[3] Metrics
------------------------------------------------------------
MAE  : 1.270586
RMSE : 1.667189
R²   : 0.611070

[4] Comparison
------------------------------------------------------------
vs Original Baseline
R² Gain  : 0.027379
MAE Gain : 0.043086

vs Advanced Dynamic Features
R² Gain  : 0.014934
MAE Gain : 0.023448

[5] Top Features
------------------------------------------------------------
                    feature  importance
49   regional_humidity_mean        3005
50    regional_humidity_std        2513
48    regional_p

In [ ]:
import lightgbm as lgb

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import numpy as np
import pandas as pd
import time

print("=" * 80)
print("SPATIAL-CONTEXT 48H FORECAST TEST")
print("=" * 80)

# ---------------------------------------------------
# Feature set
# ---------------------------------------------------
spatial_features = enhanced_features + [

    # Regional atmospheric state
    "regional_ws_mean",
    "regional_ws_std",

    "regional_pressure_mean",
    "regional_pressure_std",

    "regional_humidity_mean",
    "regional_humidity_std",

    # Relative-state features
    "ws_vs_region",
    "pressure_vs_region",
    "humidity_vs_region",

    # Normalized anomalies
    "ws_anomaly",
    "pressure_anomaly"
]

print(f"\nTotal Features: {len(spatial_features)}")

# ---------------------------------------------------
# Target
# ---------------------------------------------------
TARGET_HORIZON = 48

target_col = f"target_t_plus_{TARGET_HORIZON}"

# ---------------------------------------------------
# Dataset
# ---------------------------------------------------
temp_df = df[
    ["datetime"] + spatial_features + [target_col]
].dropna()

# ---------------------------------------------------
# Chronological split
# ---------------------------------------------------
train_df = temp_df[
    temp_df["datetime"] < "2021-01-01"
]

valid_df = temp_df[
    (temp_df["datetime"] >= "2021-01-01") &
    (temp_df["datetime"] < "2022-01-01")
]

X_train = train_df[spatial_features]
y_train = train_df[target_col]

X_valid = valid_df[spatial_features]
y_valid = valid_df[target_col]

# ---------------------------------------------------
# Model
# ---------------------------------------------------
model = lgb.LGBMRegressor(
    objective="regression",
    n_estimators=400,
    learning_rate=0.035,
    num_leaves=160,
    max_depth=14,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

# ---------------------------------------------------
# Train
# ---------------------------------------------------
print("\n[1] Training")

start = time.time()

model.fit(
    X_train,
    y_train
)

train_time = time.time() - start

print(f"Training Time: {train_time:.2f} sec")

# ---------------------------------------------------
# Predict
# ---------------------------------------------------
print("\n[2] Predicting")

preds = model.predict(X_valid)

# ---------------------------------------------------
# Metrics
# ---------------------------------------------------
print("\n[3] Metrics")
print("-" * 60)

mae = mean_absolute_error(
    y_valid,
    preds
)

rmse = np.sqrt(
    mean_squared_error(
        y_valid,
        preds
    )
)

r2 = r2_score(
    y_valid,
    preds
)

print(f"MAE  : {mae:.6f}")
print(f"RMSE : {rmse:.6f}")
print(f"R²   : {r2:.6f}")

# ---------------------------------------------------
# Historical comparison
# ---------------------------------------------------
print("\n[4] Comparison")
print("-" * 60)

baseline_r2 = 0.389178
advanced_r2 = 0.399671

baseline_mae = 1.613367
advanced_mae = 1.597361

print("vs Original Baseline")
print(f"R² Gain  : {r2 - baseline_r2:.6f}")
print(f"MAE Gain : {baseline_mae - mae:.6f}")

print("\nvs Advanced Dynamic Features")
print(f"R² Gain  : {r2 - advanced_r2:.6f}")
print(f"MAE Gain : {advanced_mae - mae:.6f}")

# ---------------------------------------------------
# Feature importance
# ---------------------------------------------------
print("\n[5] Top Features")
print("-" * 60)

importance_df = pd.DataFrame({
    "feature": spatial_features,
    "importance": model.feature_importances_
})

importance_df = (
    importance_df
    .sort_values("importance", ascending=False)
)

print(importance_df.head(20))

print("\nDone.")

SPATIAL-CONTEXT 48H FORECAST TEST

Total Features: 56

[1] Training
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.632274 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12115
[LightGBM] [Info] Number of data points in the train set: 2168605, number of used features: 56
[LightGBM] [Info] Start training from score 6.766602
Training Time: 297.10 sec

[2] Predicting

[3] Metrics
------------------------------------------------------------
MAE  : 1.604626
RMSE : 2.076453
R²   : 0.398171

[4] Comparison
------------------------------------------------------------
vs Original Baseline
R² Gain  : 0.008993
MAE Gain : 0.008741

vs Advanced Dynamic Features
R² Gain  : -0.001500
MAE Gain : -0.007265

[5] Top Features
------------------------------------------------------------
                    feature  importance
49   regional_humidity_mean        4780
50    regional_humidity_std        3999
46          re

In [ ]:
import lightgbm as lgb

from sklearn.metrics import mean_absolute_error

import numpy as np
import pandas as pd
import time

print("=" * 80)
print("48H QUANTILE FORECASTING")
print("=" * 80)

# ---------------------------------------------------
# Feature set
# ---------------------------------------------------
spatial_features = enhanced_features + [

    "regional_ws_mean",
    "regional_ws_std",

    "regional_pressure_mean",
    "regional_pressure_std",

    "regional_humidity_mean",
    "regional_humidity_std",

    "ws_vs_region",
    "pressure_vs_region",
    "humidity_vs_region",

    "ws_anomaly",
    "pressure_anomaly"
]

# ---------------------------------------------------
# Target
# ---------------------------------------------------
TARGET_HORIZON = 48

target_col = f"target_t_plus_{TARGET_HORIZON}"

# ---------------------------------------------------
# Dataset
# ---------------------------------------------------
temp_df = df[
    ["datetime"] + spatial_features + [target_col]
].dropna()

# ---------------------------------------------------
# Chronological split
# ---------------------------------------------------
train_df = temp_df[
    temp_df["datetime"] < "2021-01-01"
]

valid_df = temp_df[
    (temp_df["datetime"] >= "2021-01-01") &
    (temp_df["datetime"] < "2022-01-01")
]

X_train = train_df[spatial_features]
y_train = train_df[target_col]

X_valid = valid_df[spatial_features]
y_valid = valid_df[target_col]

# ---------------------------------------------------
# Quantiles
# ---------------------------------------------------
quantiles = [0.1, 0.5, 0.9]

models = {}
predictions = {}

# ---------------------------------------------------
# Train quantile models
# ---------------------------------------------------
for q in quantiles:

    print(f"\n{'='*60}")
    print(f"TRAINING QUANTILE: {q}")
    print(f"{'='*60}")

    model = lgb.LGBMRegressor(
        objective="quantile",
        alpha=q,
        n_estimators=300,
        learning_rate=0.04,
        num_leaves=128,
        max_depth=12,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )

    start = time.time()

    model.fit(
        X_train,
        y_train
    )

    elapsed = time.time() - start

    preds = model.predict(X_valid)

    models[q] = model
    predictions[q] = preds

    mae = mean_absolute_error(
        y_valid,
        preds
    )

    print(f"MAE: {mae:.6f}")
    print(f"Train Time: {elapsed:.2f} sec")

# ---------------------------------------------------
# Interval evaluation
# ---------------------------------------------------
print("\n" + "="*80)
print("INTERVAL EVALUATION")
print("="*80)

lower = predictions[0.1]
median = predictions[0.5]
upper = predictions[0.9]

# ---------------------------------------------------
# Coverage
# ---------------------------------------------------
inside_interval = (
    (y_valid >= lower) &
    (y_valid <= upper)
)

coverage = inside_interval.mean()

print(f"\n10%-90% Interval Coverage: {coverage:.6f}")

# ---------------------------------------------------
# Interval width
# ---------------------------------------------------
interval_width = np.mean(
    upper - lower
)

print(f"Average Interval Width: {interval_width:.6f}")

# ---------------------------------------------------
# Median forecast MAE
# ---------------------------------------------------
median_mae = mean_absolute_error(
    y_valid,
    median
)

print(f"Median Forecast MAE: {median_mae:.6f}")

# ---------------------------------------------------
# Example predictions
# ---------------------------------------------------
print("\nExample Forecast Intervals")
print("-" * 60)

examples = pd.DataFrame({
    "actual": y_valid.values[:10],
    "p10": lower[:10],
    "p50": median[:10],
    "p90": upper[:10]
})

print(examples)

print("\nDone.")

48H QUANTILE FORECASTING

TRAINING QUANTILE: 0.1
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.756742 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12115
[LightGBM] [Info] Number of data points in the train set: 2168605, number of used features: 56
[LightGBM] [Info] Start training from score 3.030000
MAE: 2.391512
Train Time: 212.49 sec

TRAINING QUANTILE: 0.5
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.397610 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12115
[LightGBM] [Info] Number of data points in the train set: 2168605, number of used features: 56
[LightGBM] [Info] Start training from score 6.900000
MAE: 1.602659
Train Time: 239.41 sec

TRAINING QUANTILE: 0.9
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.399242 seconds.
You can set `force_col_wise=true` t

In [ ]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors

print("=" * 80)
print("NEIGHBOR PROPAGATION FEATURE ENGINEERING")
print("=" * 80)

# ---------------------------------------------------
# Unique station coordinates
# ---------------------------------------------------
stations = (
    df[
        ["Index", "Latitude", "Longitude"]
    ]
    .drop_duplicates()
    .sort_values("Index")
    .reset_index(drop=True)
)

print("\n[1] Station Table")
print(stations.head())

# ---------------------------------------------------
# Remove exact duplicate coordinates
# ---------------------------------------------------
stations_unique = (
    stations
    .drop_duplicates(
        subset=["Latitude", "Longitude"]
    )
    .reset_index(drop=True)
)

print(f"\nUnique Coordinate Groups: {len(stations_unique)}")

# ---------------------------------------------------
# Nearest-neighbor model
# ---------------------------------------------------
coords = stations_unique[
    ["Latitude", "Longitude"]
].values

nbrs = NearestNeighbors(
    n_neighbors=3,
    metric="euclidean"
)

nbrs.fit(coords)

distances, indices = nbrs.kneighbors(coords)

# ---------------------------------------------------
# Build neighbor mapping
# ---------------------------------------------------
neighbor_map = {}

print("\n[2] Neighbor Mapping")

for i, row in stations_unique.iterrows():

    station_id = row["Index"]

    # Skip self (index 0)
    neighbor_idxs = indices[i][1:]

    neighbor_station_ids = (
        stations_unique
        .iloc[neighbor_idxs]["Index"]
        .tolist()
    )

    neighbor_map[station_id] = neighbor_station_ids

    print(
        f"Station {station_id} "
        f"-> Neighbors {neighbor_station_ids}"
    )

# ---------------------------------------------------
# Create neighbor lag features
# ---------------------------------------------------
print("\n[3] Creating Neighbor Features")

# Temporary lookup
base_lookup = df[
    [
        "datetime",
        "Index",
        "wind_speed_lag_1",
        "wind_speed_lag_6"
    ]
].copy()

# ---------------------------------------------------
# Neighbor feature creation
# ---------------------------------------------------
for neighbor_rank in [0, 1]:

    neighbor_feature_1 = []
    neighbor_feature_6 = []

    for idx, row in df.iterrows():

        station_id = row["Index"]

        if station_id not in neighbor_map:

            neighbor_feature_1.append(np.nan)
            neighbor_feature_6.append(np.nan)
            continue

        neighbors = neighbor_map[station_id]

        if neighbor_rank >= len(neighbors):

            neighbor_feature_1.append(np.nan)
            neighbor_feature_6.append(np.nan)
            continue

        neighbor_station = neighbors[neighbor_rank]

        dt = row["datetime"]

        match = base_lookup[
            (base_lookup["datetime"] == dt) &
            (base_lookup["Index"] == neighbor_station)
        ]

        if len(match) == 0:

            neighbor_feature_1.append(np.nan)
            neighbor_feature_6.append(np.nan)

        else:

            neighbor_feature_1.append(
                match["wind_speed_lag_1"].values[0]
            )

            neighbor_feature_6.append(
                match["wind_speed_lag_6"].values[0]
            )

    df[f"neighbor_{neighbor_rank+1}_lag1"] = (
        np.array(neighbor_feature_1)
        .astype("float32")
    )

    df[f"neighbor_{neighbor_rank+1}_lag6"] = (
        np.array(neighbor_feature_6)
        .astype("float32")
    )

print("Neighbor propagation features created.")

# ---------------------------------------------------
# Validation
# ---------------------------------------------------
print("\n[4] NaN Summary")

neighbor_cols = [
    "neighbor_1_lag1",
    "neighbor_1_lag6",
    "neighbor_2_lag1",
    "neighbor_2_lag6"
]

print(
    df[neighbor_cols]
    .isnull()
    .sum()
)

# ---------------------------------------------------
# Preview
# ---------------------------------------------------
print("\n[5] Preview")

preview_cols = [
    "Index",
    "datetime",
    "wind_speed_lag_1",
    "neighbor_1_lag1",
    "neighbor_2_lag1"
]

print(df[preview_cols].head(10))

print("\nDone.")

NEIGHBOR PROPAGATION FEATURE ENGINEERING


NameError: name 'df' is not defined